In [32]:
import os
os.listdir()

['Churn_Modelling.csv', 'Customer Churn Prediction.ipynb']

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,roc_auc_score,recall_score



#load the data
data=pd.read_csv("Churn_Modelling.csv")

#Display the first few rows of the data
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [37]:
#check for missing values
data.isnull().sum()

#Encode catagorical variables
le=LabelEncoder()
data["Geography"]=le.fit_transform(data["Geography"])
data["Gender"]=le.fit_transform(data["Gender"])

#Feature Scaling

scl=StandardScaler()
data[['CreditScore','Age','Tenure','Balance','NumOfProducts','EstimatedSalary']]=scl.fit_transform(data[['CreditScore','Age','Tenure','Balance','NumOfProducts','EstimatedSalary']])

#Drop unnecessary columns
data=data.drop(columns=['RowNumber','CustomerId','Surname'])

#Define Feature matrix X and Target vector Y

x=data.drop(columns=['Exited'])
y=data['Exited']

#Split the data into the training and testing sets

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

#Using the Logistic Regression

lr=LogisticRegression()
lr.fit(x_train,y_train)
y_pre_lr=lr.predict(x_test)

#Random Forest
rf=RandomForestClassifier()
rf.fit(x_train,y_train)
y_pre_rf=rf.predict(x_test)

#Gradient Boosting
gb=GradientBoostingClassifier()
gb.fit(x_train,y_train)
y_pre_gb=gb.predict(x_test)



In [39]:
#Function to evaluate models
def e_model(y_test,y_pre):
    print("Accuracy:",accuracy_score(y_test,y_pre))
    print("Confusion Matrix:\n",confusion_matrix(y_test,y_pre))
    print("Classification Report:\n",classification_report(y_test,y_pre))
    r_a=roc_auc_score(y_test,y_pre)
    print("ROC AUC Score:",r_a)
    return r_a

#Evaluate Logistic Regression
print("Logistic Regression:")
r_a_lr=e_model(y_test,y_pre_lr)

#Evaluate Random Forest
print("\nRandom Forest:")
r_a_rf=e_model(y_test,y_pre_rf)

#Evaluate Gradient Boosting
print("\nGradient Boosting:")
r_a_gb=e_model(y_test,y_pre_gb)

Logistic Regression:
Accuracy: 0.815
Confusion Matrix:
 [[1559   48]
 [ 322   71]]
Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.97      0.89      1607
           1       0.60      0.18      0.28       393

    accuracy                           0.81      2000
   macro avg       0.71      0.58      0.59      2000
weighted avg       0.78      0.81      0.77      2000

ROC AUC Score: 0.5753961279453282

Random Forest:
Accuracy: 0.864
Confusion Matrix:
 [[1548   59]
 [ 213  180]]
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.96      0.92      1607
           1       0.75      0.46      0.57       393

    accuracy                           0.86      2000
   macro avg       0.82      0.71      0.74      2000
weighted avg       0.85      0.86      0.85      2000

ROC AUC Score: 0.7106504462822479

Gradient Boosting:
Accuracy: 0.866
Confusion Matrix:
 [[1547   60]
 

In [41]:
# Let's assume Gradient Boosting has the highest ROC AUC score
# Hyperparameter tuning for Gradient Boosting
from sklearn.model_selection import GridSearchCV

# Define the parameter grid
param_grid={
    'n_estimators':[100,200,300],
    'learning_rate':[0.01,0.1,0.2],
    'max_depth':[3,4,5]

}

# Instantiate the grid search
grid_search=GridSearchCV(estimator=gb,param_grid=param_grid,cv=3,scoring='roc_auc',verbose=2,n_jobs=4)

# Fit the grid search to the data
grid_search.fit(x_train,y_train)

# Best performance
print("Best parameters found:",grid_search.best_params_)

# Train the optional model
best_gb=grid_search.best_estimator_
best_gb.fit(x_train,y_train)
y_pre_best_gb=best_gb.predict(x_test)

# Evaluate the optimized model
print("\nOptimized Gradient Boosting:")
r_a_best_gb=e_model(y_test,y_pre_best_gb)


Fitting 3 folds for each of 27 candidates, totalling 81 fits
Best parameters found: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}

Optimized Gradient Boosting:
Accuracy: 0.866
Confusion Matrix:
 [[1547   60]
 [ 208  185]]
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.96      0.92      1607
           1       0.76      0.47      0.58       393

    accuracy                           0.87      2000
   macro avg       0.82      0.72      0.75      2000
weighted avg       0.86      0.87      0.85      2000

ROC AUC Score: 0.7167006306695738
